# PD Model Training – XGBoost (LoanStats3a, best-practice tuning)

**Purpose:** Train a classical PD model on **LoanStats3a**-derived features from **01_lendingclub_feature_engineering.ipynb**. We use **Optuna** (Bayesian-style hyperparameter search), **stratified 5-fold CV**, and **early stopping** to maximize AUC-ROC; then refit on full data and save for `eval_runner` and proof generation (comparable to quantum models).

**Runs locally or on Google Colab.** No API keys or Colab Secrets are required (XGBoost is fully offline). On Colab, use **Runtime → Change runtime type → T4 GPU** to speed up Optuna tuning (~2–5× faster than CPU).

**Input:** `data/credit_risk_pd/LendingClub/processed/lendingclub_engineered.parquet` (from notebook 01; on Colab, run section "Setup: Colab vs local" first so the repo and data are available).

**Output:** Trained model saved to `models/pd/pd_model_local_v1.pkl`; metrics: AUC-ROC, F1, Precision, Recall. Ready for sample-by-sample evaluation.

**Tip (local):** Run from the repo root folder. Use **Kernel → Select Kernel** and pick the environment where you installed `pandas`, `xgboost`, `scikit-learn`, `optuna`.

### Install packages into this kernel's environment
Run the cell below once to see **which Python** this notebook uses. Then in a terminal (PowerShell or Bash), run:
```
<that path> -m pip install -r requirements-notebooks.txt
```
from the repo root (or `notebooks/`). That way `pip` installs into the **same** environment as this kernel.

In [9]:
import sys
print('This notebook kernel is using:')
print(sys.executable)
print()
print('To install packages into THIS environment, run in a terminal:')
print(f'  {sys.executable} -m pip install -r notebooks/requirements-notebooks.txt')
print('(from the repo root, or use a full path to requirements-notebooks.txt)')

This notebook kernel is using:
c:\Users\leemi\AppData\Local\Programs\Python\Python311\python.exe

To install packages into THIS environment, run in a terminal:
  c:\Users\leemi\AppData\Local\Programs\Python\Python311\python.exe -m pip install -r notebooks/requirements-notebooks.txt
(from the repo root, or use a full path to requirements-notebooks.txt)


## Setup: Colab vs local

Run this cell once. On **Google Colab** it clones the repo (if needed), installs dependencies, and enables **XGBoost GPU** for faster Optuna tuning. When GPU is used we set **max_bin=512** to use more GPU RAM and often speed up; on CPU we run **4 Optuna trials in parallel** to reduce wall-clock time. No API keys or Colab Secrets are required. You still need the engineered data: run **01_lendingclub_feature_engineering.ipynb** first (e.g. after uploading `LoanStats3a.csv` and running it), or upload `lendingclub_engineered.parquet` into `data/credit_risk_pd/LendingClub/processed/`. On a **local** run this cell only sets CPU mode.

In [ ]:
import os
import sys
from pathlib import Path

# Detect Colab (no API key needed for XGBoost)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
    REPO_DIR = "ocr-agentic-rag"
    os.environ["COLAB_GPU"] = os.environ.get("COLAB_GPU", "1")
    if os.path.isdir(REPO_DIR):
        get_ipython().run_line_magic("cd", REPO_DIR)
        get_ipython().system("git pull")
    else:
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
        get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("pip install -q -r notebooks/requirements-notebooks.txt")
    USE_GPU = os.environ.get("COLAB_GPU", "1") == "1"
    print("Colab: repo ready. XGBoost will use GPU for tuning." if USE_GPU else "Colab: repo ready. Using CPU (set Runtime → GPU for faster Optuna).")
else:
    USE_GPU = False
    print("Local run. For faster Optuna, open this notebook in Google Colab and set Runtime → T4 GPU.")

## 1. Load engineered data and split

In [10]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Find repo root (works when run from repo root or from notebooks/)
ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if not (ROOT / "credit_risk").is_dir():
    raise RuntimeError('Repo root not found. Run this notebook from the ocr-agentic-rag folder (or notebooks/ subfolder).')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credit_risk.feature_engineering.common_features import get_feature_names

DATA_PATH = ROOT / "data" / "credit_risk_pd" / "LendingClub" / "processed" / "lendingclub_engineered.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_lendingclub_feature_engineering.ipynb first to create lendingclub_engineered.parquet")

df = pd.read_parquet(DATA_PATH)
feature_names = get_feature_names()
X = df[[c for c in feature_names if c in df.columns]]
y = df["default"]
if X.shape[1] != len(feature_names):
    for c in feature_names:
        if c not in X.columns:
            X[c] = 0.0
X = X[feature_names]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train {X_train.shape[0]:,} / Val {X_val.shape[0]:,}")

Train 31,828 / Val 7,958


## 2. Iterative testing – XGBoost vs Random Forest

Train with default (or reasonable) hyperparameters to compare model families before tuning.

In [11]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

def eval_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc_roc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

results = {}

In [12]:
import xgboost as xgb

scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
# GPU on Colab when available; fall back to CPU if XGBoost has no GPU support (e.g. Colab default build)
xgb_kw = {"tree_method": "gpu_hist", "device": "cuda"} if USE_GPU else {"tree_method": "hist"}
if USE_GPU:
    try:
        _t = xgb.XGBClassifier(n_estimators=2, **xgb_kw)
        _t.fit(X_train[:20], y_train[:20])
    except Exception as e:
        if "gpu_hist" in str(e) or "valid values" in str(e):
            USE_GPU = False
            xgb_kw = {"tree_method": "hist"}  # no max_bin needed for CPU
        else:
            raise
model_xgb = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale,
    random_state=42,
    **xgb_kw,
)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
p_val = model_xgb.predict_proba(X_val)[:, 1]
results["XGBoost"] = eval_binary(y_val, p_val)
print("XGBoost (default):", results["XGBoost"])

XGBoost (default): {'auc_roc': 0.6647083847650476, 'f1': 0.32842105263157895, 'precision': 0.23405851462865718, 'recall': 0.5502645502645502}


In [13]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced", random_state=42)
model_rf.fit(X_train, y_train)
p_val_rf = model_rf.predict_proba(X_val)[:, 1]
results["RandomForest"] = eval_binary(y_val, p_val_rf)
print("RandomForest (default):", results["RandomForest"])
print("\nComparison:", pd.DataFrame(results).T.round(4).to_string())

RandomForest (default): {'auc_roc': 0.6632959510059939, 'f1': 0.29038388445458, 'precision': 0.25517702070808285, 'recall': 0.3368606701940035}

Comparison:               auc_roc      f1  precision  recall
XGBoost        0.6647  0.3284     0.2341  0.5503
RandomForest   0.6633  0.2904     0.2552  0.3369


## 3. Hyperparameter tuning (XGBoost) – Optuna + stratified K-fold

Best practices for LendingClub/LoanStats3a: **Bayesian-style tuning (Optuna)** with **stratified 5-fold CV** and **early stopping** to maximize AUC-ROC. We search over depth, learning rate, regularization (reg_alpha, reg_lambda, gamma), subsample/colsample, and min_child_weight; then refit the best config on full train+val for the final artifact.

In [ ]:
import time
import optuna
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Tuning config (Colab: N_TRIALS 20–50, n_jobs=1 for GPU; local CPU: n_jobs=4)
N_TRIALS = 50
OPTUNA_N_JOBS = 1 if (USE_GPU if "USE_GPU" in dir() else False) else 4
tune_start = time.perf_counter()

# GPU (e.g. Colab T4) speeds up each trial; no API key needed
xgb_common = {"tree_method": "gpu_hist", "device": "cuda"} if USE_GPU else {"tree_method": "hist"}

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-6, 1.0, log=True),
        "objective": "binary:logistic",
        "scale_pos_weight": scale,
        "eval_metric": "auc",
        "random_state": 42,
        **xgb_common,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_f, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_f, y_f, eval_set=[(X_v, y_v)], verbose=False)

        p = clf.predict_proba(X_v)[:, 1]
        aucs.append(roc_auc_score(y_v, p))
    return np.mean(aucs)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=OPTUNA_N_JOBS, show_progress_bar=True)
tune_elapsed = time.perf_counter() - tune_start

best_params = {k: v for k, v in study.best_params.items()
               if k in ["max_depth","learning_rate","n_estimators",
                        "min_child_weight","subsample","colsample_bytree",
                        "reg_alpha","reg_lambda","gamma"]}
best_params.update({
    "objective": "binary:logistic",
    "scale_pos_weight": scale,
    "eval_metric": "auc",
    "random_state": 42,
    **xgb_common,
})
print("Best params:", best_params)
print("Best CV AUC:", round(float(study.best_value), 4))
print("Tuning time: {:.1f}s ({:.1f}s per trial)".format(tune_elapsed, tune_elapsed / max(N_TRIALS, 1)))

  0%|          | 0/50 [00:00<?, ?it/s]

### 3b. Stacking ensemble (XGBoost + LightGBM) + optimal threshold

Research shows **stacking XGBoost + LightGBM** (meta-learner on their probabilities) can beat either alone on LendingClub (e.g. +6% AUC in fusion studies). We also **tune the decision threshold** for F1 (Youden’s J or max F1) instead of 0.5 to better handle class imbalance.

In [ ]:
# Run the next cell first (best_model + final_model refit on full data), then run the stacking cell in section 3c below.

In [ ]:
best_model = xgb.XGBClassifier(**best_params)
best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
p_val_best = best_model.predict_proba(X_val)[:, 1]
final_metrics = eval_binary(y_val, p_val_best)
print("Tuned XGBoost on validation:", final_metrics)

X_full = pd.concat([X_train, X_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_full, y_full, verbose=False)
# Saved model runs on CPU for eval_runner; GPU was used only for faster training.

Tuned XGBoost on validation: {'auc_roc': 0.6842322123804148, 'f1': 0.32768623810628916, 'precision': 0.22236220472440946, 'recall': 0.6225749559082893}


In [ ]:
# 3c. Stacking (XGB + LightGBM) + optimal threshold — run after refit cell above
best_threshold_opt = 0.5
p_val_best = final_model.predict_proba(X_val)[:, 1]
try:
    import lightgbm as lgb
    lgb_scale = (y_full == 0).sum() / max((y_full == 1).sum(), 1)
    lgb_params = {"objective": "binary", "metric": "auc", "boosting_type": "gbdt", "num_leaves": 31, "learning_rate": 0.05, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 5, "verbose": -1, "random_state": 42, "scale_pos_weight": lgb_scale}
    lgb_full = lgb.train(lgb_params, lgb.Dataset(X_full, y_full), num_boost_round=200)
    p_xgb_val = final_model.predict_proba(X_val)[:, 1]
    p_lgb_val = lgb_full.predict(X_val.values if hasattr(X_val, "values") else X_val)
    X_meta_train = np.column_stack([final_model.predict_proba(X_full)[:, 1], lgb_full.predict(X_full.values if hasattr(X_full, "values") else X_full)])
    meta = LogisticRegression(max_iter=500, random_state=42)
    meta.fit(X_meta_train, y_full)
    p_stacked = meta.predict_proba(np.column_stack([p_xgb_val, p_lgb_val]))[:, 1]
    metrics_stacked = eval_binary(y_val, p_stacked)
    if metrics_stacked["auc_roc"] > final_metrics["auc_roc"]:
        class _StackedPDWrapper:
            def __init__(self, xgb_model, lgb_model, meta):
                self.xgb_model, self.lgb_model, self.meta = xgb_model, lgb_model, meta
            def predict_proba(self, X):
                X_arr = X.values if hasattr(X, "values") else X
                p1 = self.xgb_model.predict_proba(X)[:, 1]
                p2 = self.lgb_model.predict(X_arr)
                p = self.meta.predict_proba(np.column_stack([p1, p2]))[:, 1]
                return np.column_stack([1 - p, p])
            def predict(self, X):
                return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)
        final_model = _StackedPDWrapper(final_model, lgb_full, meta)
        final_metrics = metrics_stacked
        p_val_best = p_stacked
        print("Stacking (XGB+LGB) beats XGBoost. Val AUC:", round(metrics_stacked["auc_roc"], 4))
    else:
        print("XGBoost kept. Stacked val AUC:", round(metrics_stacked["auc_roc"], 4))
except Exception as e:
    print("Stacking skipped:", e)
from sklearn.metrics import f1_score as f1_skl
best_f1, best_threshold_opt = 0.0, 0.5
for t in np.linspace(0.2, 0.8, 31):
    f1 = f1_skl(y_val, (p_val_best >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_threshold_opt = f1, t
print("Optimal threshold (max F1):", round(best_threshold_opt, 3), "-> F1", round(best_f1, 4))

## 4. Save model for eval_runner

Persist the tuned model in the format expected by `credit_risk.models.pd_model.PDModel` (joblib with model, feature_names, params, metadata).

In [ ]:
import joblib

MODEL_DIR = ROOT / "models" / "pd"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_data = {
    "model": final_model,
    "feature_names": feature_names,
    "params": best_params,
    "metadata": {
        "trained_at": pd.Timestamp.now().isoformat(),
        "n_features": len(feature_names),
        "n_train": len(X_full),
        "val_auc_roc": final_metrics["auc_roc"],
        "val_f1": final_metrics["f1"],
        "data_source": "LoanStats3a",
        "best_threshold": best_threshold_opt if "best_threshold_opt" in dir() else 0.5,
    },
}
path = MODEL_DIR / "pd_model_local_v1.pkl"
joblib.dump(model_data, path)
print(f"Saved to {path}")

Saved to c:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\models\pd\pd_model_local_v1.pkl


## 6. Run full evaluation (Colab or local)

**Where is ground truth?**  
- **Training** (notebook 01 + this notebook): uses **LoanStats3a.csv** → engineered to `lendingclub_engineered.parquet`.  
- **Evaluation** (this section): uses the **LendingClub benchmark** from HuggingFace (**TheFinAI/lendingclub-benchmark**), which has query + gold labels. It is stored as **parquet** under `data/credit_risk_pd/LendingClub/{test,train,valid}/`. If you use a script like `data/generate_pq_first_5_rows.py` (or the repo’s download script), it downloads from the Hub and can export parquet + first_5_rows for LendingClub.

On **Colab**, the cell below ensures the evaluation parquet exists (downloads from HF if needed), runs `eval_runner.py --category credit_risk_PD --dataset LendingClub` with the saved model, then displays the evaluation summary.

In [ ]:
# Ensure LendingClub evaluation data exists (parquet under test/train/valid); then run eval and show summary
import json
import subprocess

LC_BASE = ROOT / "data" / "credit_risk_pd" / "LendingClub"
PROOF_PD = ROOT / "data" / "proof" / "credit_risk_pd"

def _parquet_exists():
    for split in ("test", "train", "valid"):
        folder = LC_BASE / split
        if not folder.exists():
            return False
        if not list(folder.glob("*.parquet")):
            return False
    return True

if not _parquet_exists():
    print("LendingClub evaluation parquet not found. Downloading from HuggingFace (TheFinAI/lendingclub-benchmark)...")
    try:
        from datasets import load_dataset
        ds = load_dataset("TheFinAI/lendingclub-benchmark")
        split_map = {"train": "train", "test": "test", "validation": "valid"}
        for hf_split, folder_name in split_map.items():
            if hf_split not in ds:
                continue
            out_dir = LC_BASE / folder_name
            out_dir.mkdir(parents=True, exist_ok=True)
            path = out_dir / "data-00000-of-00001.parquet"
            ds[hf_split].to_parquet(path)
            print(f"  Wrote {path} ({len(ds[hf_split])} rows)")
        print("Done.")
    except Exception as e:
        print("Download failed:", e)
        print("Ensure data/credit_risk_pd/LendingClub/{test,train,valid}/*.parquet exist (e.g. run scripts/download_datasets.py and convert to parquet per split).")
else:
    print("LendingClub evaluation data found.")

# Run eval_runner for credit_risk_PD (uses saved model at models/pd/pd_model_local_v1.pkl)
print("\nRunning eval_runner.py --category credit_risk_PD --dataset LendingClub ...")
result = subprocess.run(
    [sys.executable, str(ROOT / "eval_runner.py"), "--category", "credit_risk_PD", "--dataset", "LendingClub"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print("eval_runner stderr:", result.stderr[:1500] if result.stderr else "(none)")
    print("eval_runner stdout:", result.stdout[:1500] if result.stdout else "(none)")
else:
    print("Eval completed.")

In [ ]:
# Display evaluation summary from proof outputs
summary_shown = False
# Dataset-level weighted average (LendingClub)
weighted_path = PROOF_PD / "lendingclub" / "lendingclub_weighted_avg.json"
if weighted_path.exists():
    with open(weighted_path, "r", encoding="utf-8") as f:
        w = json.load(f)
    print("=== PD evaluation summary (LendingClub benchmark) ===\n")
    print("Dataset weighted average:")
    for k, v in sorted(w.items()):
        if isinstance(v, (int, float)):
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
        else:
            print(f"  {k}: {v}")
    summary_shown = True
# Per-split averages
for split_dir in (PROOF_PD / "lendingclub").iterdir() if (PROOF_PD / "lendingclub").exists() else []:
    if not split_dir.is_dir():
        continue
    avg_path = split_dir / "lendingclub_{}_avg.json".format(split_dir.name)
    if avg_path.exists():
        with open(avg_path, "r", encoding="utf-8") as f:
            s = json.load(f)
        auc = s.get("auc_roc_mean")
        f1 = s.get("f1_mean")
        n = s.get("sample_count", 0)
        auc_s = f"{auc:.4f}" if isinstance(auc, (int, float)) else str(auc)
        f1_s = f"{f1:.4f}" if isinstance(f1, (int, float)) else str(f1)
        print(f"\nSplit '{split_dir.name}' avg: auc_roc={auc_s}  f1={f1_s}  n={n}")
        summary_shown = True
# Category-level (credit_risk_pd)
cat_path = ROOT / "data" / "proof" / "credit_risk_pd_weighted_avg.json"
if cat_path.exists():
    with open(cat_path, "r", encoding="utf-8") as f:
        c = json.load(f)
    print("\nCategory (credit_risk_pd) weighted avg:", json.dumps(c, indent=2))
    summary_shown = True
if not summary_shown:
    print("No evaluation summary files found. Run the cell above to run eval_runner and generate proofs.")

### 5b. Training report (copy for feedback)

Run the cell below after tuning and saving. **Copy the block between the dashed lines** and paste it into chat so the assistant can suggest hyperparameter or training improvements for the next run.

In [ ]:
# Key metrics report for iterative tuning (copy the output below the dashes)
try:
    _te = tune_elapsed
except NameError:
    _te = None
_report = [
    "--- PD XGBoost training report (copy for feedback) ---",
    "env: colab={} | gpu_used={} | optuna_n_jobs={}".format(
        IN_COLAB if "IN_COLAB" in dir() else False,
        USE_GPU,
        OPTUNA_N_JOBS if "OPTUNA_N_JOBS" in dir() else 1,
    ),
    "data: n_train={} n_val={} n_features={}".format(
        len(X_train), len(X_val), X_train.shape[1],
    ),
    "tuning: n_trials={} n_splits={}".format(
        N_TRIALS if "N_TRIALS" in dir() else 50,
        N_SPLITS if "N_SPLITS" in dir() else 5,
    ),
]
if _te is not None and "N_TRIALS" in dir():
    _report.append("timing: total_tune_sec={:.1f} sec_per_trial={:.1f}".format(_te, _te / max(N_TRIALS, 1)))
else:
    _report.append("timing: (not recorded)")
_report.extend([
    "best_cv_auc: {:.4f}".format(study.best_value),
    "val_metrics: auc_roc={:.4f} f1={:.4f} precision={:.4f} recall={:.4f}".format(
        final_metrics["auc_roc"], final_metrics["f1"],
        final_metrics["precision"], final_metrics["recall"],
    ),
    "best_params: " + " | ".join("{}={}".format(k, v) for k, v in best_params.items()),
    "---",
])
print("\n".join(_report))

## 5. Summary

- Trained XGBoost (and Random Forest) on LoanStats3a-derived features; tuned XGBoost via **Optuna** (stratified 5-fold CV, early stopping) to maximize AUC-ROC.
- Final model refit on full train+val and saved as `models/pd/pd_model_local_v1.pkl`. For **overnight sample-by-sample evaluation** on a CPU-only machine (e.g. i5-11500, 16GB), run from repo root: `python eval_runner.py --category credit_risk_PD`. Proof JSONs are written under `data/proof/credit_risk_pd/` for comparison with quantum models later.